# Project: Handwritten Digit Recognition with PyTorch

**Learning goal:** Build a small neural network that recognizes handwritten digits (0–9), and understand the complete machine-learning workflow.

**What you'll learn**
1. What a dataset, feature, label, training set and test set are.
2. How images become tensors.
3. What a neural network, loss function and optimizer do.
4. How to train on a GPU (when one is available).
5. How to evaluate a model and inspect its mistakes.

**Time:** approximately 15–25 minutes, depending on runtime hardware and download speed.

> This is a teaching project, not a production-grade model. Run the cells from top to bottom. Read each explanation before running its code.

## 0. Choose a GPU runtime (optional but recommended)

In Colab, select **Runtime → Change runtime type → T4 GPU** if that option is available, then click Save.

A GPU is not required for this small project. The code automatically uses a GPU when available and otherwise falls back to the CPU.

**Concept:** A GPU can perform many numerical operations in parallel. Neural-network training often benefits from this, although a tiny model like ours may not run dramatically faster on a GPU.

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np

# Choose the GPU if Colab assigned one; otherwise use the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No GPU assigned. CPU training will work too.")

## 1. Load the MNIST dataset

MNIST contains 70,000 small grayscale images of handwritten digits:
- 60,000 images for training.
- 10,000 images for testing.
- Each image is 28 × 28 pixels.
- Each image has a label from 0 to 9.

A **feature** is information the model uses to make a prediction. Here, the pixel intensities are the features.

A **label** is the correct answer. Here, the label is the digit shown in the image.

We normalize pixel values from the range 0–255 to a more convenient range around 0–1.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# A batch is a small group of examples processed together.
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))
print("Number of training batches:", len(train_loader))

## 2. Look at the data

Before training a model, inspect examples. This is a basic but important data-science habit.

The image is a 28 × 28 grid of grayscale pixel values. The label tells us which digit it represents.

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis("off")

plt.tight_layout()
plt.show()

print("One image tensor shape:", images[0].shape)
print("Pixel range:", images.min().item(), "to", images.max().item())

## 3. Build a neural network

We'll use a **fully connected neural network** (also called a multilayer perceptron).

Our architecture:
1. Input: 28 × 28 = 784 pixel values.
2. Hidden layer: 128 neurons with ReLU activation.
3. Output layer: 10 scores, one for each possible digit.

Why 10 outputs? Because the model must choose among ten classes (0–9).

The output values are called **logits**. We'll pass them directly to `CrossEntropyLoss`; do not apply softmax before that loss function.

In [ ]:
import torch.nn as nn

class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),          # [batch, 1, 28, 28] -> [batch, 784]
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)     # Ten class scores (logits)
        )

    def forward(self, x):
        return self.network(x)

model = DigitClassifier().to(device)

print(model)

total_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_parameters:,}")

## 4. Choose the loss function and optimizer

Two important training components:

- **Loss function:** measures how far predictions are from the correct labels. We'll use cross-entropy, a standard choice for multi-class classification.
- **Optimizer:** updates model parameters to reduce the loss. We'll use Adam.

The learning rate controls the approximate size of the optimizer's update steps. We'll start with `0.001`.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", loss_function)
print("Optimizer:", optimizer.__class__.__name__)

## 5. Train the model

An **epoch** means one complete pass through the training dataset.

For each batch, the training loop:
1. Moves images and labels to the selected device.
2. Runs a forward pass to get predictions.
3. Calculates the loss.
4. Clears old gradients.
5. Computes new gradients with backpropagation.
6. Updates the model parameters.

We'll train for three epochs to keep the first experiment short. You can increase this later.

In [ ]:
def train_one_epoch(model, loader, loss_function, optimizer, device):
    model.train()  # Enable training behavior
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # 1. Forward pass
        logits = model(images)

        # 2. Calculate loss
        loss = loss_function(logits, labels)

        # 3. Backpropagation and parameter update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        total_loss += loss.item() * images.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


EPOCHS = 3
history = {"train_loss": [], "train_accuracy": []}

for epoch in range(EPOCHS):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, loss_function, optimizer, device
    )
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Accuracy: {train_accuracy * 100:.2f}%"
    )

## 6. Evaluate on unseen test images

Training accuracy alone is not enough. A model might memorize its training examples.

The test dataset contains examples the model did not train on. Evaluating against it gives us a better estimate of how well the model generalizes.

We'll use `model.eval()` and `torch.no_grad()` because we are evaluating, not updating weights.

In [ ]:
def evaluate(model, loader, loss_function, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = loss_function(logits, labels)

            total_loss += loss.item() * images.size(0)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


test_loss, test_accuracy = evaluate(model, test_loader, loss_function, device)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

## 7. Visualize predictions

Let's inspect individual predictions. Green titles indicate a correct prediction; red titles indicate a mistake.

A model's errors are useful: they can show which digits are visually confusing.

In [ ]:
model.eval()
images, labels = next(iter(test_loader))

with torch.no_grad():
    logits = model(images.to(device))
    predictions = logits.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    true_label = labels[i].item()
    predicted_label = predictions[i].item()
    is_correct = true_label == predicted_label

    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(
        f"Predicted: {predicted_label}\nActual: {true_label}",
        color="green" if is_correct else "red"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Save your trained model

Training takes time, so save the learned parameters. A `state_dict` stores the model's weights.

In Colab, files saved in the runtime are temporary. Download this file or copy it to Google Drive if you want to keep it after the session ends.

In [ ]:
model_path = "mnist_digit_classifier.pth"
torch.save(model.state_dict(), model_path)
print("Saved model to:", model_path)

# Optional: download the model file to your computer.
from google.colab import files
files.download(model_path)

## 9. Experiments: change one thing at a time

Now that the baseline works, try these small experiments. Record the test accuracy after each change.

**Experiment A — Train longer**
- Change `EPOCHS = 3` to `EPOCHS = 5`.
- Does test accuracy improve?

**Experiment B — Change the learning rate**
- Re-create the model and optimizer.
- Try `lr=0.01`, then `lr=0.0001`.
- Does training become unstable or slower?

**Experiment C — Increase model capacity**
- Change the hidden layer from 128 to 256 neurons.
- Does it improve test accuracy? Does it take longer?

**Important:** When comparing experiments, create a fresh model and optimizer each time. Otherwise, you're continuing from previously learned weights.

## 10. Check your understanding

1. What is the difference between a training dataset and a test dataset?
2. What does one epoch mean?
3. Why do we call `optimizer.zero_grad()` before `loss.backward()`?
4. What does `model.eval()` change?
5. Why do we move both the model and the input tensors to the same device?
6. If training accuracy is high but test accuracy is much lower, what might be happening?

**Suggested next step:** Try Experiment A, note the test accuracy, and then tell me your result. We can interpret it together and move on to a small CNN that learns spatial patterns in images.